## Imports & Global Variables

In [1]:
%matplotlib widget
import rospy
import actionlib
import threading
import time
from nav_msgs.msg     import Odometry
from sensor_msgs.msg  import LaserScan
from assignment_2_2024.msg  import PlanningAction, PlanningGoal 
from assignment2_rt_ros.msg import RobotState, Target 
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np

robot_state = RobotState()
min_obstacle_distance = None
vis_pos = None
vis_goals = None
ani_pos = None
ani_goals = None

## Topic Callbacks

In [2]:
def odom_callback(msg): 
    # Every time new data is available on the /odom topic update and publish the robot state
    robot_state.x = msg.pose.pose.position.x                      
    robot_state.y = msg.pose.pose.position.y
    robot_state.vel_x = msg.twist.twist.linear.x
    robot_state.ang_vel_z = msg.twist.twist.angular.z     
    pub_state.publish(robot_state)

def scan_callback(msg):
    # Every time new data is available on the /scan topic extract the smallest distance 
    global min_obstacle_distance
    valid = [r for r in msg.ranges if msg.range_min < r < msg.range_max]
    min_obstacle_distance = min(valid) if valid else None

## Action Finished Callback

In [3]:
def done_callback(state, result):
    # Print feedback when a goal is reached or canceled and update the counts of reached/not reached goals
    with status_out:
        status_out.clear_output()
        if state == actionlib.GoalStatus.SUCCEEDED:
            print("Goal reached!")
            vis_goal.update_counts(True)
        elif state == actionlib.GoalStatus.PREEMPTED:
            print("Goal canceled!")
            vis_goal.update_counts(False)
        else:
            print(f"Goal failed (status {state}).")
            vis_goal.update_counts(False)

## Visualizer Classes

In [4]:
class PositionVisualizer:
    # Initialize plot and relative settings
    def __init__(self):
        # With plt.ioff and plt.ion we disable temporarily the auto-draw so that the plots are not shown right after the UI 
        # initialization, but only at the end in the final UI (otherwise we would have doubled plots)
        plt.ioff()
        self.fig_pos, self.ax_pos = plt.subplots(figsize=(4.3,4.3))
        plt.ion()
        self.ln_pos, = self.ax_pos.plot([], [], 'bo-', markersize=5, label='Robot Path')
        self.x_data, self.y_data = [], []
        self.ax_pos.set_xlabel("X Position (m)")
        self.ax_pos.set_ylabel("Y Position (m)")
        self.ax_pos.set_title("Robot Position")
        self.ax_pos.legend()
        self.ax_pos.grid(True)
        self.ax_pos.set_xlim(-10, 10)
        self.ax_pos.set_ylim(-10, 10)

    def plot_init_pos(self):
        self.x_data.clear()
        self.y_data.clear()
        self.ln_pos.set_data([], [])
        self.ax_pos.set_xlim(-10, 10)
        self.ax_pos.set_ylim(-10, 10)
        return self.ln_pos,

    def update_plot_pos(self, frame):
        # Append current robot position (retrieved by robot_state variable, which is updated by the odom callback) if it's valid
        # Only append if the position has actually changed or if it's the first point
        if robot_state.x is not None and robot_state.y is not None:
            if not self.x_data or (self.x_data[-1] != robot_state.x or self.y_data[-1] != robot_state.y):
                self.x_data.append(robot_state.x)
                self.y_data.append(robot_state.y)
        self.ln_pos.set_data(self.x_data, self.y_data)
        return self.ln_pos,
    

class GoalStatusVisualizer:
    def __init__(self):
        plt.ioff()
        self.fig_goals, self.ax_goals = plt.subplots(figsize=(4.3,4.3))
        plt.ion()
        self.categories = ['Reached', 'Not Reached']
        self.counts = [0, 0]                # [reached_count, not_reached_count]
        self.bars = self.ax_goals.bar(self.categories, self.counts, color=['green', 'red'])
        self.ax_goals.set_ylabel("Count")
        self.ax_goals.set_title("Goal Status")
        self.ax_goals.set_ylim(0, 5)

    def plot_init_goals(self):
        self.counts = [0,0]
        for bar, count in zip(self.bars, self.counts):
            bar.set_height(count)
        self.ax_goals.set_ylim(0, 5)
        return self.bars

    def update_counts(self, reached: bool):
        # Update the counts of reached/not reached goals
        if reached:
            self.counts[0] += 1
        else:
            self.counts[1] += 1

    def update_plot_goals(self, frame):
        # Update plot with new values and resize it if needed
        for bar, count in zip(self.bars, self.counts):
            bar.set_height(count)

        # Adjust y-axis limit dynamically
        max_count = max(self.counts) if any(c > 0 for c in self.counts) else 1    # Ensure max_count is at least 1
        current_ylim_top = self.ax_goals.get_ylim()[1]
        if max_count >= current_ylim_top:
            self.ax_goals.set_ylim(0, max_count + max(1, int(max_count * 0.2)))   # Add 20% padding or at least 1
        elif max_count < current_ylim_top * 0.7 and current_ylim_top > 5 :        # Shrink if too much space (but not below 5)
             self.ax_goals.set_ylim(0, max_count + max(1, int(max_count * 0.2)))

        self.ax_goals.figure.canvas.draw_idle()
        return self.bars

## UI Initialization

In [5]:
pos_label     = widgets.Label("Position: N/A")
vel_label = widgets.Label("Velocity: N/A")
obstacle_label  = widgets.Label("Closest obstacle: N/A")

x_input       = widgets.FloatText(description='Target X:', value=0.0)
y_input       = widgets.FloatText(description='Target Y:', value=0.0)
send_button   = widgets.Button(description='Send Goal',   button_style='success')
cancel_button = widgets.Button(description='Cancel Goal', button_style='danger')

status_out = widgets.Output()

vis_pos = PositionVisualizer()
vis_goal = GoalStatusVisualizer()

ani_pos = FuncAnimation(vis_pos.fig_pos, vis_pos.update_plot_pos,
                        init_func=vis_pos.plot_init_pos, frames=None, interval=200, blit=True, save_count=0)

ani_goals = FuncAnimation(vis_goal.fig_goals, vis_goal.update_plot_goals,
                          init_func=vis_goal.plot_init_goals, frames=None, interval=500, blit=True, save_count=0)

pos_canvas  = vis_pos.fig_pos.canvas
pos_canvas.header_visible = False
goal_canvas = vis_goal.fig_goals.canvas
goal_canvas.header_visible = False
goal_canvas.footer_visible = False

send_box = widgets.HBox(
    [x_input, y_input, send_button, cancel_button],
    layout=widgets.Layout(
        border='1px solid lightgray',
        padding='10px',
        margin='5px 0',
        align_items='stretch'
    )
)

status_box = widgets.VBox(
    [status_out],
    layout=widgets.Layout(
        border='1px solid lightgray',
        padding='10px',
        margin='5px 0',
        min_height='50px',
        align_items='stretch'
    )
)

info_box = widgets.VBox(
    [pos_label, vel_label, obstacle_label],
    layout=widgets.Layout(
        border='1px solid lightgray',
        padding='10px',
        margin='5px 0',
        align_items='stretch'
    )
)

plots_box = widgets.HBox(
    [pos_canvas, goal_canvas], 
    layout=widgets.Layout(
        border='1px solid lightgray',
        padding='10px',
        margin='5px 0',
        align_items='stretch'
    )
)

ui = widgets.VBox([
    send_box,
    status_box,
    info_box,
    plots_box
], layout=widgets.Layout(
    width='100%',
    align_items='stretch')
)

## UI Callbacks

In [6]:
def on_send(_):
    with status_out:
        status_out.clear_output()
        
        # Before sending a goal we need to cancel any pending or active goal
        state = client.get_state()
        if state in (actionlib.GoalStatus.PENDING, actionlib.GoalStatus.ACTIVE):
            client.cancel_goal()
            # Wait until the goal is really gone otherwise we would get error
            while client.get_state() in (actionlib.GoalStatus.PENDING, actionlib.GoalStatus.ACTIVE):
                time.sleep(0.05)

        # Send the new goal
        goal = PlanningGoal()
        goal.target_pose.pose.position.x = x_input.value
        goal.target_pose.pose.position.y = y_input.value
        client.send_goal(goal, done_cb=done_callback)

        # Publish target to the /last_target topic as we did in the RT1 assignment
        pub_target.publish(Target(x=x_input.value, y=y_input.value))

        print(f"Goal sent : ({x_input.value:.2f}, {y_input.value:.2f})")

def on_cancel(_):
    with status_out:
        status_out.clear_output()
        
        # Here we need to ensure that there is an active/pending goal to cancel in order to not get errors
        st = client.get_state()
        if st in (actionlib.GoalStatus.PENDING, actionlib.GoalStatus.ACTIVE):
            client.cancel_goal()
        else:
            print("No active goal to cancel.")

# Link callbacks to buttons
send_button.on_click(on_send)
cancel_button.on_click(on_cancel)

## ROS setup

In [7]:
def setup_ros():
    global client, pub_state, pub_target
    
    rospy.init_node('jupyter_go_to_point', anonymous=True)

    # Subscribers & publishers
    rospy.Subscriber('/odom',  Odometry,   odom_callback)
    rospy.Subscriber('/scan',  LaserScan,  scan_callback)
    pub_state  = rospy.Publisher('/robot_state', RobotState, queue_size=10)
    pub_target = rospy.Publisher('/last_target',  Target,     queue_size=10)

    # Action client
    client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
    rospy.loginfo("Waiting for action server…")
    client.wait_for_server()
    rospy.loginfo("Connected to action server.")   
    
setup_ros()

[INFO] [1747046259.929653, 6671.980000]: Waiting for action server…
[INFO] [1747046259.970634, 6672.022000]: Connected to action server.


## Feedback Setup

In [8]:
# Start feedback thread
def update_ui():
    rate = 5.0         # Hertz 
    while not rospy.is_shutdown():
        # We need to update the UI with the values from robot_state
        pos_label.value = (
            f"Position: (x: {robot_state.x:.2f}, y: {robot_state.y:.2f})"
        )
        vel_label.value = (
            f"Velocity: (linear: {robot_state.vel_x:.2f}, angular: {robot_state.ang_vel_z:.2f})"
        )
        if min_obstacle_distance is not None:
            obstacle_label.value = f"Closest obstacle: {min_obstacle_distance:.2f} m"
        else:
            obstacle_label.value = "Closest obstacle: N/A"
        time.sleep(1.0 / rate)

def start_update_ui():
    t = threading.Thread(target=update_ui, daemon=True)
    t.start()
    
start_update_ui()

## UI Display

In [9]:
display(ui)